# 03 · Experimentos (Colab T4)

Corre el diseño experimental completo: **23 corridas** (línea base, factorial LoRA, extremos y DoRA) con reanudación.

**Cómo usarlo:**
1. Abrir este notebook en Google Colab (ver el README: *Reproducir los experimentos*).
2. **Forzar GPU T4:** Entorno de ejecución → Cambiar tipo de entorno → T4, para que todas las corridas sean comparables.
3. Ejecutar las celdas en orden. La celda de *Setup* clona el repositorio, instala el paquete y monta Google Drive (pide autorización); no hay que descomentar nada.
4. Cada corrida guarda su fila en `MyDrive/triaje/results/resultados.csv` apenas termina. Si Colab se desconecta, volver a ejecutar todas las celdas: la suite salta las corridas ya hechas y continúa.

Duración medida: ~6,5 min por corrida LoRA y ~13,5 min por corrida DoRA, unas **3 h** en total.

In [ ]:
# === Setup ===
# En Google Colab: clona la rama del proyecto, instala el paquete y monta Drive.
# Fuera de Colab (local, desde la raíz del repo) esta celda no hace nada.
import os, sys
EN_COLAB = "google.colab" in sys.modules
RAMA = "splits-con-manual"  # cambiar a "main" cuando la rama se fusione
if EN_COLAB:
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone -q -b {RAMA} https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    # Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16); no lo usamos.
    !pip uninstall -q -y torchao
    # Colab ya trae las dependencias: esto solo verifica versiones mínimas e instala `triaje`.
    !pip install -q -r requirements.txt
    !pip install -q -e .
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
if EN_COLAB:
    # Resultados y adaptadores a Drive: sobreviven si Colab se desconecta, y la
    # suite reanuda leyendo el resultados.csv de Drive.
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir = config.drive_dir / "adaptadores"
set_seed(config.seed)
print("modelo:", config.model_name, "| resultados en:", config.results_dir)

## Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU. Activa la T4 en Colab."
print("GPU:", torch.cuda.get_device_name(0))

## (Opcional) Smoke-test antes de gastar GPU

Verifica que el pipeline corre con un modelo diminuto y pocos ejemplos.

In [ ]:
# Descomentar para probar el pipeline de punta a punta en ~2 min (modelo diminuto,
# 10 ejemplos). Guarda en results_smoke/, así que no interfiere con la suite real.
# from triaje.experimentos import correr_suite
# smoke = Config().aplicar_smoke_test()
# correr_suite(smoke, rs_piloto=(), rs_factorial=(4,), ratios=(1,), semillas=(42,),
#              incluir_piloto=False, dora_configs=((4, 1),))

## Correr la suite completa

23 corridas, en este orden: baseline; ronda 1 (semilla 42) con los extremos r ∈ {2, 128} y el factorial r ∈ {4, 16, 64} × α/r ∈ {1, 2}; ronda 2 (semilla 123) con lo mismo; y al final DoRA (α/r=2, r ∈ {4, 16, 64}) con ambas semillas. Revisa el tiempo de las primeras corridas para estimar el total.

In [ ]:
from triaje.experimentos import correr_suite
correr_suite(config, incluir_baseline=True)   # reanuda automáticamente